In [11]:
import pandas as pd
oecd_data = pd.read_csv('/Users/riley/Desktop/Data Case Competition/oecd_cleaned.csv')
country_coords = pd.read_csv('https://raw.githubusercontent.com/google/dspl/master/samples/google/canonical/countries.csv')

In [12]:
country_coords.head()

,country,latitude,longitude,name
0,AD,42.546245,1.601554,Andorra
1,AE,23.424076,53.847818,United Arab Emirates
2,AF,33.939110,67.709953,Afghanistan
3,AG,17.060816,-61.796428,Antigua and Barbuda
4,AI,18.220554,-63.068615,Anguilla


In [13]:
oecd_data[['Donor_country', 'country', 'usd_disbursements_defl']].head()

,Donor_country,country,usd_disbursements_defl
0,United Kingdom,Afghanistan,0.007338
1,United Kingdom,Kenya,0.008166
2,United Kingdom,Ukraine,6.795787
3,United Kingdom,Ukraine,0.339789
4,United Kingdom,Ukraine,4.077472


In [14]:
flow_data = oecd_data.merge(
    country_coords[['name', 'latitude', 'longitude']],
    left_on='country',
    right_on='name',
    how='left'
)

flow_data = flow_data.rename(columns={
    'latitude': 'recipient_lat',
    'longitude': 'recipient_lon'
})

flow_data = flow_data.merge(
    country_coords[['name', 'latitude', 'longitude']],
    left_on='Donor_country',
    right_on='name',
    how='left'
)

flow_data = flow_data.rename(columns={
    'latitude': 'donor_lat',
    'longitude': 'donor_lon'
})

flow_data = flow_data.drop(columns=['Unnamed: 0', 'expected_duration', 'name_x', 'name_y'])
flow_data.head()


,year,usd_disbursements_defl,country,region,region_macro,organization_name,Donor_country,broad_sector,recipient_lat,recipient_lon,donor_lat,donor_lon
0,2021,0.007338,Afghanistan,South & Central Asia,Asia,Lund Trust,United Kingdom,humanitarian_and_disaster,33.939110,67.709953,55.378051,-3.435973
1,2021,0.008166,Kenya,Eastern Africa,Africa,Lund Trust,United Kingdom,environment_and_agriculture,-0.023559,37.906193,55.378051,-3.435973
2,2022,6.795787,Ukraine,Europe,Europe,Lund Trust,United Kingdom,humanitarian_and_disaster,48.379433,31.165580,55.378051,-3.435973
3,2022,0.339789,Ukraine,Europe,Europe,Lund Trust,United Kingdom,other,48.379433,31.165580,55.378051,-3.435973
4,2022,4.077472,Ukraine,Europe,Europe,Lund Trust,United Kingdom,humanitarian_and_disaster,48.379433,31.165580,55.378051,-3.435973


In [ ]:
flow_data = flow_data.dropna(subset=['donor_lat', 'donor_lon', 'recipient_lat', 'recipient_lon'])
flow_data_grouped = flow_data.groupby(
    ['Donor_country', 'country', 'year', 'broad_sector', 'donor_lat', 'donor_lon', 'recipient_lat', 'recipient_lon'],
    as_index=False
)['usd_disbursements_defl'].sum()

In [16]:
flow_data_grouped.shape

(5665, 9)

In [17]:
flow_data_grouped.to_csv('tableau_data.csv')